In [1]:
import os
import sys
from lca_algebraic import initProject
import pandas as pd
import numpy as np
import xarray as xr
import lca_algebraic as agb
from lca_algebraic import *  # version 1.0.5 
from lca_algebraic.stats import * 
import bw2data, bw2io
import bw2calc
from premise import *
import premise as prm
import matplotlib.pyplot as plt
import matplotlib
from matplotlib import colormaps
matplotlib.style.use('seaborn-v0_8-whitegrid') #seaborn-v0_8-whitegrid  tableau-colorblind10

In [2]:
sys.path.append('/Users/susierwu/Dropbox/DLCA_DynCO2/pdpLCA_GT/') 
from utils import *
from utils.setup import *
from utils.getAct import *
from utils.utils import *
#from utils.switchElec import *

modify the setup.py file for reading Parameter excel sheet. here we use 'data/p_file/p_tech_perSSP_Y.xlsx' as the technology parameter file.  
Define your chosen methods in the setup.py file. Current methods are:  [('IPCC 2021 no LT', 'climate change no LT', 'global warming potential (GWP100) no LT'), ('IPCC 2021', 'climate change', 'GWP 100a, incl. H and bio CO2')]
Mapping premise_remind_DB to SSPx:  {'SSP1-PkBudg500': 'ssp119', 'SSP2-Base': 'ssp245', 'SSP5-Base': 'ssp585'}
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
pGWP100 are imported as:  [('IPCC 2021 - dpCFsSSP119_MY2030 - year100', 'climate change', 'pGWP100'), ('IPCC 2021 - dpCFsSSP245_MY2030 - year100', 'climate change', 'pGWP100'), ('IPCC 2021 - dpCFsSSP585_MY2030 - year100', 'climate change', 'pGWP100'), ('IPCC 2021 - dpCFsSSP119_MY2040 - year100', 'climate change', 'pGWP100'), ('IPCC 2021 - dpCFsSSP245_MY2040 - year100', 'climate change', 'pGWP100'), ('IPCC 2021

In [3]:
from utils_pGWP.pGWP_F2_V1 import *

In [4]:
bw2data.projects.set_current('iveo_v1')
mybio = bw2data.Database("biosphere3")
initProject("iveo_v1")

Biosphere database already present!!! No setup is needed


### init 

In [5]:
root_path = '/Users/susierwu/Dropbox/DLCA_DynCO2/pdpLCA_GT/'
p_file = os.path.abspath(os.path.join(root_path, 'data/P_files/p_tech_perSSP_Y.xlsx'))

In [6]:
pdb = bw.Database("iveo_Parameterized_v1")
print(len(pdb))
agb.loadParams()

384


[ParamRegistry] Param NMC622_battery_cell_density was already defined in '<project>' : overriding.
[ParamRegistry] Param NCA_battery_cell_density was already defined in '<project>' : overriding.
[ParamRegistry] Param LFP_battery_cell_density was already defined in '<project>' : overriding.
[ParamRegistry] Param LTO_battery_cell_density was already defined in '<project>' : overriding.
[ParamRegistry] Param LSB_battery_cell_density was already defined in '<project>' : overriding.
[ParamRegistry] Param LSB_cell_assmb_NG_MJperkg was already defined in '<project>' : overriding.
[ParamRegistry] Param LSB_cell_assmb_elec_kWhperkg was already defined in '<project>' : overriding.
[ParamRegistry] Param g_truck_vehicle_prod_lightweighting was already defined in '<project>' : overriding.
[ParamRegistry] Param battery_total_capacity_for_gtruck was already defined in '<project>' : overriding.
[ParamRegistry] Param biodiesel_UCO_perc was already defined in '<project>' : overriding.
[ParamRegistry] Pa

## A. prepare all P and UPRs


#### B.0 test opening Parameter file

##### <font color = "red"> use function loadYearlyParams_multisheet() in setup.py :: need to readin yearlyParams from different sheet as g_truck BEV version also depending on battery-related parameters </font>  

## C. adding switch parameter 

all URP prepared in notebook under "notebooks/notebook_tech_veh/"

#### using the switch_act in final assemly UPR, updatingabove 2.2

# D. final calc with all P

## get already created UPR

In [9]:
gtruck_finalwhbat_switch_allsspyear  = [] 
for prm_name in ssp_scn:
    for year_t in ssp_yr: 
        bg_db_ssp_yr = bg_ei_ + prm_name + "_" + str(year_t)            # backgroun database (BD) to use for SSP[x], Year[t]
        copied_prm_UPR_suffix = "_from_" + prm_name + "_" + str(year_t) # copy UPR from BD to PDB for further use/modification if needed
        newUPR_suffix =  "_wh_" + prm_name + "_" + str(year_t)          # naming on any new UPRs created in PDB
        # final one unit of g_truck with lightweighting as Parameter: 
        gtruck_final  = agb.findActivity(
                db_name = PDB_NAME,
                name="V1A_2_g-truck BEV_final_assem_wh_switch_bat" + newUPR_suffix)
        gtruck_finalwhbat_switch_allsspyear.append(gtruck_final)

len(gtruck_finalwhbat_switch_allsspyear)

9

### D1. read in TRL table to see which switch_P to apply

In [10]:
data = xr.open_dataset('../../data/to_use_interm/TRL_1C_vehDSwhTRL.nc')
data2 = data.where(data["tech_avail"] == 1, drop = True) 
#data2.sel( powertrain = "BEV", Module = "V1B",  SSP = "ssp119", v_year = 2040)

In [12]:
### to print out TRL
tech_todolca_list,tech_ssp_list, tech_year_list, tech_score_list = [],[],[],[]
for gtruck_final_whSwitchbat in gtruck_finalwhbat_switch_allsspyear:
    # need a matched YearlyParams for each SSPx:
    str1 = gtruck_final_whSwitchbat.as_dict()["name"]
    str2 = str1.split("remind_")[1]
    year_to_get_inP = int( str2.split("_")[-1] )
    ssp_remind = str2.split("_")[0] 
    ssp_to_get_inP = ssp_remind_Pname_map.get(ssp_remind)
    #print(f"get available tech lsit for SSP {ssp_to_get_inP} year {year_to_get_inP}")
    
    all_app_bat = []  
    for bat_x in ["LIB_NMC622", 'LIB_LFP',  'LIB_LTO', 'LIB_NCA',  "LSB"]:  
        bat_avail = data2.sel(Tech = bat_x, powertrain = "BEV", Module = "V1B", v_type = "garbage_truck",  
                              SSP = ssp_to_get_inP, v_year = year_to_get_inP)["tech_avail"].values
        tech_todolca_list.append(bat_x)
        tech_ssp_list.append(ssp_to_get_inP)
        tech_year_list.append(year_to_get_inP)
        
        if np.isnan(bat_avail).all(): 
            print(f"not available for {bat_x} per SSP {ssp_to_get_inP} year {year_to_get_inP}")
            tech_score_list.append(0)
        else: 
            tech_score_list.append(1)
            
            all_app_bat.append(bat_x)
            #print(all_app_bat)
  
all_tech_todolca = pd.DataFrame(zip(tech_todolca_list,tech_ssp_list, tech_year_list, tech_score_list), 
                               columns = ["tech","ssp","year","score"])

all_tech_todolca

not available for LSB per SSP ssp119 year 2030
not available for LSB per SSP ssp245 year 2030
not available for LSB per SSP ssp585 year 2030
not available for LSB per SSP ssp585 year 2040
not available for LSB per SSP ssp585 year 2050


,tech,ssp,year,score
0,LIB_NMC622,ssp119,2030,1
1,LIB_LFP,ssp119,2030,1
2,LIB_LTO,ssp119,2030,1
3,LIB_NCA,ssp119,2030,1
4,LSB,ssp119,2030,0
5,LIB_NMC622,ssp119,2040,1
6,LIB_LFP,ssp119,2040,1
7,LIB_LTO,ssp119,2040,1
8,LIB_NCA,ssp119,2040,1
9,LSB,ssp119,2040,1


#### D.1.0 testing read all P file 

In [11]:
root_path = '/Users/susierwu/Dropbox/DLCA_DynCO2/pdpLCA_GT/'
p_file = os.path.abspath(os.path.join(root_path, 'data/P_files/p_tech_perSSP_Y.xlsx'))

for gtruck_final_whSwitchbat in gtruck_finalwhbat_switch_allsspyear:
    # need a matched YearlyParams for each SSPx:
    str1 = gtruck_final_whSwitchbat.as_dict()["name"]
    str2 = str1.split("remind_")[1]
    year_to_get_inP = int( str2.split("_")[-1] )
    ssp_remind = str2.split("_")[0] 
    ssp_to_get_inP = ssp_remind_Pname_map.get(ssp_remind)
    
    yearlyParams = loadYearlyParams_multisheet(PARAMS_FILE = p_file, s_name = ['V1A_g_truck', 'V1B_bat_LSB_perkWh', 
                                              'V1B_bat_prmLIBs_perkWh'], years = [year_to_get_inP], SSP = ssp_to_get_inP) 
    print(f"for SSP {ssp_to_get_inP}, year {year_to_get_inP}, P:  {yearlyParams}" )

for SSP ssp119, year 2030, P:  {'g_truck_vehicle_prod_lightweighting': [260.0], 'battery_total_capacity_for_gtruck': [376.0], 'LSB_battery_cell_density': [0.5], 'LSB_cell_assmb_NG_MJperkg': [5.47], 'LSB_cell_assmb_elec_kWhperkg': [1.38], 'NMC622_battery_cell_density': [0.29161000000000004], 'NCA_battery_cell_density': [0.30358900000000005], 'LFP_battery_cell_density': [0.21054000000000003], 'LTO_battery_cell_density': [0.21054000000000003]}
for SSP ssp119, year 2040, P:  {'g_truck_vehicle_prod_lightweighting': [314.6], 'battery_total_capacity_for_gtruck': [346.5216], 'LSB_battery_cell_density': [0.6565], 'LSB_cell_assmb_NG_MJperkg': [2.88], 'LSB_cell_assmb_elec_kWhperkg': [0.65], 'NMC622_battery_cell_density': [0.3528481000000001], 'NCA_battery_cell_density': [0.36734269000000014], 'LFP_battery_cell_density': [0.2547534000000001], 'LTO_battery_cell_density': [0.2547534000000001]}
for SSP ssp119, year 2050, P:  {'g_truck_vehicle_prod_lightweighting': [380.6660000000001], 'battery_total_

## D.2 calculate with the bat_tech_switch P and yearlyP, 
-   IPCC_GWP100 | premise_GWP100 | pGWP100

In [13]:
#final output to be appended
gtruck_lcia_dict = {}
GWP0_list, GWP1_list, GWP2_list, SSP_list, Year_list, tech_list = [], [],[],[],[],[]

for gtruck_final_whSwitchbat in gtruck_finalwhbat_switch_allsspyear:
    #print(gtruck_final_whSwitchbat)
    # need a matched YearlyParams for each SSPx:
    str1 = gtruck_final_whSwitchbat.as_dict()["name"]
    str2 = str1.split("remind_")[1]
    year_to_get_inP = int( str2.split("_")[-1] )
    ssp_remind = str2.split("_")[0] 
    ssp_to_get_inP = ssp_remind_Pname_map.get(ssp_remind)
    
    yearlyParams = loadYearlyParams_multisheet(PARAMS_FILE = p_file, s_name = ['V1A_g_truck', 'V1B_bat_LSB_perkWh', 'V1B_bat_prmLIBs_perkWh'], years = [year_to_get_inP], SSP = ssp_to_get_inP) 
    #print(f"for SSP {ssp_to_get_inP}, year {year_to_get_inP}, P:  {yearlyParams}" )

    """ prepare methods: GWP1_list using static GWP,  GWP2_list using the pGWP100 for the matched v_year """ 
    gwp_spx_yt = get_my_pGWP100(ssp = ssp_to_get_inP, my = year_to_get_inP, metric = 'pGWP') 
    methods_touse = [chosen_methods[0], chosen_methods[1], gwp_spx_yt] # IPCCGWP100, premiseGWP, pGWP100_per_premiseGWP

    """ see what tech can be used under SSPx   """
    all_app_bat = []  
    for bat_x in ["LIB_NMC622",  'LIB_LFP',  'LIB_LTO', 'LIB_NCA', "LSB"]:
        bat_avail = data2.sel(Tech = bat_x, powertrain = "BEV", Module = "V1B", v_type = "garbage_truck",  
                              SSP = ssp_to_get_inP, v_year = year_to_get_inP)["tech_avail"].values
        if np.isnan(bat_avail).all(): 
            pass
        else: 
            all_app_bat.append(bat_x)
    print(f" @@@@@@ for SSP {ssp_to_get_inP} year {year_to_get_inP}, available {all_app_bat}")

    for b_tech in all_app_bat :
        if b_tech == "LIB_NMC622":    #mapping names in data2, to be updated 
            b_tech = "NMC622"
        elif b_tech == "LIB_LFP":
            b_tech = "LFP"
        elif b_tech == "LIB_LTO":
            b_tech = "LTO"
        elif b_tech == "LIB_NCA": 
            b_tech = "NCA"
    
        df= agb.multiLCAAlgebric(models=[
                (gtruck_final_whSwitchbat, 1)],  #act_test
                methods = methods_touse,   
                #define switch parameter here if you have one, otherwise, using the default value, e.g., assuming assembly @CA_QC 
                gtruck_BEV_battery_tech = b_tech ,
                **yearlyParams
            )

        print(f"end of running LCA calc for SSP {ssp_to_get_inP} and year {year_to_get_inP} using battery tech {b_tech} for {gtruck_final_whSwitchbat} ")
        display(df)
        
        # prepare for final DF: 
        GWP0 = df.iloc[:,0].values[0]
        GWP1 = df.iloc[:,1].values[0]
        GWP2 = df.iloc[:,2].values[0]
        GWP0_list.append(GWP0)
        GWP1_list.append(GWP1)
        GWP2_list.append(GWP2)
        SSP_list.append(ssp_to_get_inP) 
        Year_list.append(year_to_get_inP)
        tech_list.append(b_tech)
        
        # assign df as value to each UPR name (used as key)
        gtruck_lcia_dict[str1] = df
    

 @@@@@@ for SSP ssp119 year 2030, available ['LIB_NMC622', 'LIB_LFP', 'LIB_LTO', 'LIB_NCA']
end of running LCA calc for SSP ssp119 and year 2030 using battery tech NMC622 for 'V1A_2_g-truck BEV_final_assem_wh_switch_bat_wh__remind_SSP1-PkBudg500_2030' (one_truck, GLO, None) 


,climate change no LT - global warming potential (GWP100) no LT[kg CO2-Eq],"climate change - GWP 100a, incl. H and bio CO2[kg CO2-Eq]",climate change - pGWP100[kg CO2-Eq]
V1A_2_g-truck BEV_final_assem_wh_switch_bat_wh__remind_SSP1-PkBudg500_2030,177826.645604,182908.570336,179060.507296


end of running LCA calc for SSP ssp119 and year 2030 using battery tech LFP for 'V1A_2_g-truck BEV_final_assem_wh_switch_bat_wh__remind_SSP1-PkBudg500_2030' (one_truck, GLO, None) 


,climate change no LT - global warming potential (GWP100) no LT[kg CO2-Eq],"climate change - GWP 100a, incl. H and bio CO2[kg CO2-Eq]",climate change - pGWP100[kg CO2-Eq]
V1A_2_g-truck BEV_final_assem_wh_switch_bat_wh__remind_SSP1-PkBudg500_2030,167226.777071,170649.031771,167228.071545


end of running LCA calc for SSP ssp119 and year 2030 using battery tech LTO for 'V1A_2_g-truck BEV_final_assem_wh_switch_bat_wh__remind_SSP1-PkBudg500_2030' (one_truck, GLO, None) 


...

,climate change no LT - global warming potential (GWP100) no LT[kg CO2-Eq],"climate change - GWP 100a, incl. H and bio CO2[kg CO2-Eq]",climate change - pGWP100[kg CO2-Eq]
V1A_2_g-truck BEV_final_assem_wh_switch_bat_wh__remind_SSP5-Base_2050,156577.020758,160377.50019,157749.910889


end of running LCA calc for SSP ssp585 and year 2050 using battery tech LTO for 'V1A_2_g-truck BEV_final_assem_wh_switch_bat_wh__remind_SSP5-Base_2050' (one_truck, GLO, None) 


,climate change no LT - global warming potential (GWP100) no LT[kg CO2-Eq],"climate change - GWP 100a, incl. H and bio CO2[kg CO2-Eq]",climate change - pGWP100[kg CO2-Eq]
V1A_2_g-truck BEV_final_assem_wh_switch_bat_wh__remind_SSP5-Base_2050,186259.686268,192228.431851,189224.380594


end of running LCA calc for SSP ssp585 and year 2050 using battery tech NCA for 'V1A_2_g-truck BEV_final_assem_wh_switch_bat_wh__remind_SSP5-Base_2050' (one_truck, GLO, None) 


,climate change no LT - global warming potential (GWP100) no LT[kg CO2-Eq],"climate change - GWP 100a, incl. H and bio CO2[kg CO2-Eq]",climate change - pGWP100[kg CO2-Eq]
V1A_2_g-truck BEV_final_assem_wh_switch_bat_wh__remind_SSP5-Base_2050,159807.848034,167052.210502,164374.363336


### prepare DF and DS 

In [22]:
d1 = get_plcia_wh_V1AV1B_BEV(SSP_list, Year_list, tech_list, GWP0_list, GWP1_list, GWP2_list)[0]
d2 = get_plcia_wh_V1AV1B_BEV(SSP_list, Year_list, tech_list, GWP0_list, GWP1_list, GWP2_list)[1]

In [23]:
d2

IPCC2021_default    premise_GWP  \
SSP    v_module  v_year tech_list                                    
ssp119 V1AB_init 2030   NMC622        177826.645604  182908.570336   
                        LFP           167226.777071  170649.031771   
                        LTO           200100.835461  205924.481911   
                        NCA           170804.973391  178041.387302   
                 2040   NMC622        147318.151655  150467.783374   
                        LFP           139244.731125  141130.270365   
                        LTO           164283.352788  167997.919001   
                        NCA           141970.074703  146760.679338   
                        LSB           128219.084880  129934.014084   
                 2050   NMC622        129484.917428  130434.605013   
                        LFP           123335.773329  123322.661220   
                        LTO           142406.511945  143786.483520   
                        NCA           125411.539148  127611.078501   
                        LSB           114190.579859  114041.666237   
ssp245 V1AB_init 2030   NMC622        194366.659435  199780.165484   
                        LFP           183766.790902  187520.626919   
                        LTO           216640.849292  222796.077060   
                        NCA           187344.987222  194912.982450   
                 2040   NMC622        167237.315587  172293.310350   
                        LFP           158823.998447  162562.681559   
                        LTO           184916.763470  190561.476860   
                        NCA           161664.080882  168430.134659   
                        LSB           150161.761963  153744.804167   
                 2050   NMC622        146869.223322  151643.580205   
                        LFP           140191.413585  143920.195171   
                        LTO           160901.737162  166143.371210   
                        NCA           142445.640998  148577.304226   
                        LSB           133945.106752  137555.281483   
ssp585 V1AB_init 2030   NMC622        207218.725922  212768.722460   
                        LFP           195585.310117  199313.809386   
                        LTO           231664.775561  238028.815889   
                        NCA           199512.400862  207426.961488   
                 2040   NMC622        195350.881701  200824.046621   
                        LFP           184799.030631  188620.043834   
                        LTO           217524.169355  223735.695991   
                        NCA           188361.017702  195978.911953   
                 2050   NMC622        166147.860732  171446.890473   
                        LFP           156577.020758  160377.500190   
                        LTO           186259.686268  192228.431851   
                        NCA           159807.848034  167052.210502   

                                         pGWP100  
SSP    v_module  v_year tech_list                 
ssp119 V1AB_init 2030   NMC622     179060.507296  
                        LFP        167228.071545  
                        LTO        201989.634646  
                        NCA        174523.363763  
                 2040   NMC622     148202.722438  
                        LFP        139101.172751  
                        LTO        165683.605863  
                        NCA        144668.116242  
                        LSB        128014.377164  
                 2050   NMC622     128667.065631  
                        LFP        121718.193737  
                        LTO        141983.778186  
                        NCA        125957.563620  
                        LSB        112523.797369  
ssp245 V1AB_init 2030   NMC622     194890.342713  
                        LFP        183125.323799  
                        LTO        217807.477174  
                        NCA        190415.135903  
                 2040   NMC622     169486.722384  
                       

In [24]:
BEV_init_ds = get_plcia_ds(d2)
BEV_init_ds

<xarray.Dataset>
Dimensions:          (SSP: 3, v_module: 1, v_year: 3, tech_list: 5,
                      Impact_category: 3)
Coordinates:
  * SSP              (SSP) object 'ssp119' 'ssp245' 'ssp585'
  * v_module         (v_module) object 'V1AB_init'
  * v_year           (v_year) int64 2030 2040 2050
  * tech_list        (tech_list) object 'LFP' 'LSB' 'LTO' 'NCA' 'NMC622'
  * Impact_category  (Impact_category) object 'IPCC2021_default' ... 'premise...
Data variables:
    value            (SSP, v_module, v_year, tech_list, Impact_category) float64 ...

In [25]:
BEV_init_ds.to_netcdf('pGWP_module_output/V1AB_init_BEVGT_pGWP100.nc')

In [26]:
d1.to_excel('pGWP_module_output/excel_output/V1AB_init_BEVGT_pGWP100_NMConly.xlsx' )
d2.to_excel('pGWP_module_output/excel_output/V1AB_init_BEVGT_pGWP100_alltech.xlsx' )